# 01 — Images and Masks

Before any correlation, you need two things:
- An **`Image`** — a greyscale photograph with pre-computed interpolation data.
- A **`Mask`** — defining which pixels to include in a subset or which region to analyse.

This notebook covers: `Image`, `Mask` (local and global), `CircleRegion`, `PathRegion`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopyv_dev as gp

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "animation.embed_limit": 50,
})


In [ ]:
# Generate a reference and a deformed speckle image.
# 5 px x-translation + slight shear — fast enough to run in < 2 s.
os.makedirs("data/single", exist_ok=True)
speckle = gp.Speckle(
    image_cfg={"image_dir": "data/single", "name": "demo", "image_size": (401, 401)},
    speckle_cfg={"speckle_size": 3.5, "speckle_number": 700},
    progression="deformation",
    deformation_cfg={"comp": [5.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
    noise_cfg=(0.0, 1.5),
    scale_cfg={"scale": "lin", "n": 2},
)
speckle.solve(seed=42)
f_img = gp.Image(filepath="data/single/demo_0.jpg")
g_img = gp.Image(filepath="data/single/demo_1.jpg")
print(f"Reference : {f_img}")
print(f"Target    : {g_img}")


## The `Image` class

`Image` loads a JPEG/PNG, converts to greyscale, applies a mild Gaussian blur
(σ = 1.1), and pre-computes the **bi-quintic B-spline coefficient matrix** (`qcqt`).
This matrix lets the DIC solver evaluate sub-pixel intensity values extremely quickly
without recomputing the spline at every iteration.

```python
img = gp.Image(filepath="photo.jpg")   # from file
img = gp.Image(image_gs=array)         # from numpy array (already greyscale)
```

Key attributes:

| Attribute | Shape | Description |
|-----------|-------|-------------|
| `image_gs` | (H, W) | Greyscale pixel values 0–255 |
| `qcqt` | (H·6, W·6) | Pre-computed B-spline coefficient blocks |
| `border` | int | Padding used for coefficient computation |
| `filepath` | str \| None | Source path |


In [ ]:
# Explore the Image object
print(f_img)
print(f"  image_gs shape : {f_img.image_gs.shape}")
print(f"  qcqt shape     : {f_img.qcqt.shape}")
print(f"  border         : {f_img.border}")
print(f"  filepath       : {f_img.filepath}")


In [ ]:
# Display the reference image alongside a zoomed crop
fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(11, 4.5))

img_arr = f_img.image_gs
ax_full.imshow(img_arr, cmap="gray", vmin=0, vmax=255)
ax_full.set_title("Full reference image", fontsize=12, fontweight="bold")
ax_full.axis("off")

# Zoom into the centre — shows individual speckle blobs
cx, cy, R = 200, 200, 60
ax_zoom.imshow(img_arr[cy-R:cy+R, cx-R:cx+R], cmap="gray",
               extent=[cx-R, cx+R, cy+R, cy-R], vmin=0, vmax=255)
rect = patches.Rectangle((cx-R, cy-R), 2*R, 2*R, lw=2, ec="crimson", fc="none")
ax_full.add_patch(rect)
ax_zoom.set_title("Zoomed crop (120 × 120 px)", fontsize=12, fontweight="bold")
ax_zoom.set_xlabel("x (px)"); ax_zoom.set_ylabel("y (px)")
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
fig.savefig("../docs/assets/nb1_image_zoom.png", dpi=120, bbox_inches="tight")
plt.show()


### Under the hood: B-spline interpolation

The DIC solver must evaluate intensities at **sub-pixel** positions (e.g. x = 127.43).
`Image` pre-computes the B-spline coefficient matrix `qcqt` so that any sub-pixel
lookup is just a small matrix multiply — no recomputation needed per iteration.

The block for pixel (i, j) lives at `qcqt[i*6 : i*6+6, j*6 : j*6+6]`.


In [ ]:
# Visualise the B-spline coefficient block for one pixel
# and show how it enables smooth sub-pixel interpolation.
import matplotlib.gridspec as gridspec
from scipy.interpolate import RectBivariateSpline

px, py = 140, 140   # pixel we're zooming into
W = 4               # window radius (pixels)
crop = f_img.image_gs[py-W:py+W+1, px-W:px+W+1]

# Build a smooth interpolant over the crop using scipy for illustration
xi = np.arange(px-W, px+W+1, dtype=float)
yi = np.arange(py-W, py+W+1, dtype=float)
interp = RectBivariateSpline(yi, xi, crop)
xf = np.linspace(px-W, px+W, 120)
yf = np.linspace(py-W, py+W, 120)
Zf = interp(yf, xf)

fig = plt.figure(figsize=(11, 4.5))
gs = gridspec.GridSpec(1, 2, wspace=0.35)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Pixel grid
im1 = ax1.imshow(crop, cmap="gray", vmin=0, vmax=255,
                  extent=[px-W-0.5, px+W+0.5, py+W+0.5, py-W-0.5], interpolation="nearest")
ax1.scatter(*np.meshgrid(xi, yi), s=10, c="red", zorder=5, label="pixel centres")
ax1.set_title("Pixel grid (discrete samples)", fontsize=11, fontweight="bold")
ax1.set_xlabel("x (px)"); ax1.set_ylabel("y (px)")
ax1.legend(fontsize=9)
fig.colorbar(im1, ax=ax1, fraction=0.046, label="Intensity")

# Smooth surface
im2 = ax2.imshow(Zf, cmap="gray", vmin=0, vmax=255,
                  extent=[px-W, px+W, py+W, py-W])
ax2.scatter(*np.meshgrid(xi, yi), s=10, c="red", zorder=5)
ax2.set_title("B-spline surface (sub-pixel)", fontsize=11, fontweight="bold")
ax2.set_xlabel("x (px)"); ax2.set_ylabel("y (px)")
fig.colorbar(im2, ax=ax2, fraction=0.046, label="Intensity")
plt.suptitle("From discrete pixels to a smooth intensity surface", fontsize=12, y=1.02)
fig.savefig("../docs/assets/nb1_bspline.png", dpi=120, bbox_inches="tight")
plt.show()


## The `Mask` class

A `Mask` defines which pixels belong to a subset.  There are two fundamentally
different types, both accessed through the same class via `mask_type`:

| `mask_type` | Purpose | Key parameters |
|-------------|---------|----------------|
| `"local"` | Pixel offsets relative to a subset centre | `shape` (`"circle"` / `"square"`), `size` |
| `"global"` | Binary image-space mask (entire ROI at once) | `f_img`, `boundary`, `exclusions` |

> **Rule of thumb**: use `"local"` for `Subset` and `Mesh` (the default).
> Use `"global"` only when you need pixel-level masking of the image itself.


In [ ]:
# --- Local masks: circle vs square ---
mask_c = gp.Mask(mask_type="local", shape="circle", size=25)
mask_s = gp.Mask(mask_type="local", shape="square", size=25)
print(mask_c)
print(mask_s)
print(f"Circle: {mask_c.n_px} active pixels  |  Square: {mask_s.n_px} active pixels")


In [ ]:
# Visualise local mask pixel grids
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
SIZE = 25

for ax, mask, title in [
    (axes[0], mask_c, f"Local circle  (size={SIZE}, n_px={mask_c.n_px})"),
    (axes[1], mask_s, f"Local square  (size={SIZE}, n_px={mask_s.n_px})"),
]:
    # Build a 2D grid from the flat coordinate offsets
    coords = mask.coords          # shape (n_px, 2) — [dx, dy] offsets
    sm     = mask.subset_mask     # shape (2*size+1, 2*size+1) binary grid
    ax.imshow(sm, cmap="Blues", vmin=-0.2, vmax=1.2, interpolation="nearest")

    # Overlay active pixel markers
    ys, xs = np.where(sm)
    ax.scatter(xs, ys, s=6, c="navy", alpha=0.5, linewidths=0)

    # Mark centre
    c = SIZE
    ax.plot(c, c, "r+", ms=12, mew=2, label="centre")

    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=9)

plt.suptitle("Local mask templates — active pixels in blue", fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig("../docs/assets/nb1_local_masks.png", dpi=120, bbox_inches="tight")
plt.show()


### Global masks

A **global mask** computes a binary image of shape (H, W) based on a boundary polygon
and optional exclusion regions.  Pixels inside the boundary (and outside exclusions)
are set to 1 (active); all others are 0.

Global masks are passed to `Subset` or created standalone with `Mask(mask_type="global")`.


In [ ]:
# Define a boundary and an exclusion region
boundary_nodes = np.array([
    [50.0, 50.0], [50.0, 350.0], [350.0, 350.0], [350.0, 50.0]
])
excl_nodes = np.array([
    [280.0, 280.0], [280.0, 340.0], [340.0, 340.0], [340.0, 280.0]
])

global_mask = gp.Mask(
    mask_type="global",
    f_img=f_img,
    boundary=gp.PathRegion(nodes=boundary_nodes, hard=True),
    exclusions=[gp.PathRegion(nodes=excl_nodes, hard=True)],
)
print(global_mask)
print(f"Binary mask shape: {global_mask.binary.shape}")
print(f"Active pixels: {global_mask.binary.sum()} / {global_mask.binary.size}")


In [ ]:
# Show the global mask overlaid on the image
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].imshow(f_img.image_gs, cmap="gray", vmin=0, vmax=255)
axes[0].add_patch(patches.Polygon(boundary_nodes[:, [0, 1]], closed=True,
                                   lw=2, ec="crimson", fc="crimson", alpha=0.15))
axes[0].add_patch(patches.Polygon(excl_nodes[:, [0, 1]], closed=True,
                                   lw=2, ec="royalblue", fc="royalblue", alpha=0.3))
axes[0].set_title("Boundary (red) + exclusion (blue)", fontsize=11, fontweight="bold")
axes[0].axis("off")

masked = f_img.image_gs.copy().astype(float)
masked[global_mask.binary == 0] = np.nan
im = axes[1].imshow(masked, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Image after global mask", fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
fig.savefig("../docs/assets/nb1_global_mask.png", dpi=120, bbox_inches="tight")
plt.show()


## `CircleRegion` and `PathRegion`

Regions are **tracked** boundary/exclusion objects.  Unlike plain numpy arrays, they:
- Record their full deformation history (`history_nodes`, `history_centres`)
- Can be updated when the reference image advances in a `Sequence`

The `option` parameter controls how the region moves between frames:

| `option` | Meaning |
|----------|---------|
| `"F"` | **Flexible** — region deforms with the local mesh warp (default) |
| `"R"` | **Rigid** — region translates rigidly |
| `"S"` | **Static** — region never moves |
| `"D"` | **Displacement-only** — translates by mean displacement |


In [ ]:
# CircleRegion: circle centred at (200, 200), radius 80 px
circle_region = gp.CircleRegion(centre=[200.0, 200.0], radius=80.0, size=15.0, option="F")
print(circle_region)
print(f"  Nodes shape : {circle_region.current_nodes.shape}")
print(f"  Centre      : {circle_region.current_centre}")
print(f"  hard        : {circle_region.hard}")

# PathRegion: arbitrary polygon
path_region = gp.PathRegion(nodes=boundary_nodes, option="R")
print(path_region)


In [ ]:
# Visualise both region types on the image
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(f_img.image_gs, cmap="gray", vmin=0, vmax=255, alpha=0.7)

cn = circle_region.current_nodes
ax.fill(cn[:, 0], cn[:, 1], fc="crimson", ec="crimson", alpha=0.2, lw=2, label="CircleRegion")
ax.plot(np.append(cn[:, 0], cn[0, 0]), np.append(cn[:, 1], cn[0, 1]), "r-", lw=2)

pn = path_region.current_nodes
ax.fill(pn[:, 0], pn[:, 1], fc="royalblue", ec="royalblue", alpha=0.15, lw=2, label="PathRegion")
ax.plot(np.append(pn[:, 0], pn[0, 0]), np.append(pn[:, 1], pn[0, 1]), "b-", lw=2)

ax.legend(fontsize=10); ax.axis("off")
ax.set_title("CircleRegion vs PathRegion on the speckle image", fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig("../docs/assets/nb1_regions.png", dpi=120, bbox_inches="tight")
plt.show()


## Summary

| Object | When to use |
|--------|------------|
| `Image` | Always — wraps every photograph |
| `Mask(mask_type="local")` | Passed to `Subset`, `Mesh`, `Sequence` as the subset template |
| `Mask(mask_type="global")` | When you need pixel-level masking of the image |
| `CircleRegion` | Circular boundary or exclusion that should track the deformation |
| `PathRegion` | Polygonal boundary or exclusion with optional tracking |

**Next:** Notebook 02 — correlating a single subset to find a point displacement.
